# RSNA Knee - queue worker on Colab (DRAFT, 11 Oct, not yet run)

Runs the same queue worker the RunPod pods ran (`runpod/mega_boot.sh` + `runpod/mega.py` from the private dataset
`girishbose/rsna-knee-mega-queue`): it downloads the dense80 cache from our Kaggle kernel outputs, trains every arm
listed in the queue's `arms.jsonl` with `train_knee.py`, and uploads each finished arm as the private dataset
`girishbose/rsna-knee-rp-<job>-out`, which `scripts/build_own_probe.py` / `build_ensemble_kernel.py` already resolve.

Before running:
1. Runtime -> Change runtime type -> a GPU (L4 or A100; T4 is the Kaggle speed).
2. Key icon (Secrets) -> add `KAGGLE_API_TOKEN` (the token from the Mac's `~/.kaggle/access_token`) and allow this notebook to read it. The token is never printed.
3. On the Mac: `python scripts/mega.py unstop`, then `python scripts/mega.py add --round <R> --labels <labels>.parquet --families <family> --folds 0,1,2,3,4` for the arms to train.

Known limits: the trainer cannot resume - a disconnected session loses the arm in progress (finished arms are already uploaded). Keep the tab open; one arm per GPU.

In [ ]:
from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
open(os.path.expanduser('~/.kaggle/access_token'), 'w').write(os.environ['KAGGLE_API_TOKEN'])
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
print('token set:', bool(os.environ['KAGGLE_API_TOKEN']))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1
!mkdir -p /content/ws/out /content/ws/queue && ln -sfn /content/ws /workspace
!pip install -q kaggle
!kaggle datasets download girishbose/rsna-knee-mega-queue -p /workspace/queue --unzip --force -q && ls /workspace/queue | head -30

In [ ]:
# the pod boot script, unchanged: installs deps, fetches the 40 GB dense80 cache (~10-15 min), then runs the queue worker.
# Without RUNPOD_* variables its self-delete calls do nothing. Progress: /workspace/out/job.log and <job>/<tag>/train.log
!cd /workspace/out && bash /workspace/queue/mega_boot.sh

In [ ]:
!tail -5 /workspace/out/job.log; cat /workspace/out/STATUS; ls /workspace/out